In [42]:
import pandas as pd
import unicodedata
import re
import lxml


To analyze historical NBA draft performance, I will collect draft data from Basketball Reference covering the 2010–2025 NBA Drafts.

Pandas' `read_html()` function will allow us to extracts HTML tables directly from web pages and converts them into DataFrames.

In [43]:
# Collect draft data from Basketball Reference
all_drafts = []

for year in range(2010, 2026):
    url = f"https://www.basketball-reference.com/draft/NBA_{year}.html"
    df = pd.read_html(url)[0]

    # Flatten MultiIndex columns
    df.columns = df.columns.get_level_values(1)

    df["DraftYear"] = year
    all_drafts.append(df)

drafts = pd.concat(all_drafts, ignore_index=True)

# Remove repeated header rows and round divider rows
drafts = drafts[pd.to_numeric(drafts["Pk"], errors="coerce").notna()].copy()
drafts["Pk"] = drafts["Pk"].astype(int)

drafts.to_csv("nba_draft_data.csv", index=False)

To prepare for merging, player names from the combine dataset are converted from "Last, First" format to "First Last" format so they match the naming convention used in the draft dataset. The same is done for draft year.

In [44]:
# Load and clean combine data
combine = pd.read_csv("NBA_Draft_Combine_2000_2024_nl.csv")

def last_first_to_first_last(name):
    parts = str(name).split(", ")

    if len(parts) == 2:
        return f"{parts[1]} {parts[0]}"

    return name

combine["Player"] = combine["Player Name (Last Name, First Name)"].apply(
    last_first_to_first_last
)

combine = combine.rename(columns={"Year": "DraftYear"})

combine = combine[
    combine["DraftYear"].between(2010, 2025)
].copy()

combine.to_csv("combine_data.csv", index=False)

In [45]:
# Better name normalization for matching
def normalize_name(name):
    name = str(name)

    # Remove accents: Vásquez -> Vasquez
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")

    name = name.lower()

    # Remove suffixes: Jr., III, IV, etc.
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\.?\b", "", name)

    # Remove punctuation and weird characters
    name = re.sub(r"[^a-z ]", " ", name)

    # Collapse extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

drafts["Player_clean"] = drafts["Player"].apply(normalize_name)
combine["Player_clean"] = combine["Player"].apply(normalize_name)

Merge our draft and combine datasets on Player name and Year that they were drafted.

Note: Draft records were matched to NBA Combine records by normalized player name and draft year. Unmatched rows mostly represent players who did not appear in the combine dataset, including international players, players who skipped the combine, or records with alternate names/nicknames.

In [46]:
# Merge using cleaned player name + draft year
master = drafts.merge(
    combine,
    on=["Player_clean", "DraftYear"],
    how="left",
    suffixes=("_draft", "_combine")
)

matched = master["Position"].notna().sum()

print(f"Matched: {matched}")
print(f"Total draft picks: {len(master)}")
print(f"Match Rate: {matched / len(master):.2%}")

Matched: 661
Total draft picks: 956
Match Rate: 69.14%


This section downloads player-level college basketball statistics from the public toRvik dataset for seasons 2014–2023. Three categories of statistics are collected:

- Advanced metrics
- Traditional box score statistics
- Shooting statistics


In [51]:
def load_torvik_player_season(year, stat):
    url = f"https://raw.githubusercontent.com/andreweatherman/toRvik-data/main/player_season/{year}/{stat}_{year}.parquet"
    df = pd.read_parquet(url)
    df["year"] = year
    return df

years = range(2014, 2024)  # 2014 through 2023

advanced = pd.concat(
    [load_torvik_player_season(year, "advanced") for year in years],
    ignore_index=True
)

box = pd.concat(
    [load_torvik_player_season(year, "box") for year in years],
    ignore_index=True
)

shooting = pd.concat(
    [load_torvik_player_season(year, "shooting") for year in years],
    ignore_index=True
)

advanced.to_csv("college_advanced_2014_2023.csv", index=False)
box.to_csv("college_box_2014_2023.csv", index=False)
shooting.to_csv("college_shooting_2014_2023.csv", index=False)

Match rows using the same player id and the same college year. 

The same player could appear in multiple seasons, so you don't want to merge only on id. We want the right player and the right season.

In [56]:
# Explore potential columns to merge on
set(advanced.columns) & set(box.columns) & set(shooting.columns)

merge_keys = ["id", "year"]

# Keep every row from advanced, even if there is no match in box or shooting. The suffixes part handles duplicate column names.
college = (
    advanced
    .merge(box, on=merge_keys, how="left", suffixes=("", "_box"))
    .merge(shooting, on=merge_keys, how="left", suffixes=("", "_shooting"))
)

college.to_csv("college_stats_2014_2023.csv", index=False)

To connect college performance data with NBA draft outcomes, drafted players were matched to their corresponding college statistics using standardized player names.

Since many players appear in the college dataset multiple times (one row per season), a matching process was used to identify the most relevant college season for each draft prospect. Candidate matches were restricted to seasons occurring before or during the player's draft year.


In [58]:
college["Player_clean"] = college["player"].apply(normalize_name)

# Give every draft/combine row a stable ID before merging college stats
master = master.reset_index(drop=True).copy()
master["draft_row_id"] = master.index

# Only college data years you actually have
draft_for_college = master[
    master["DraftYear"].between(2014, 2023)
    & master["Player_draft"].notna()
].copy()

# Match each drafted player to all possible college seasons
college_candidates = draft_for_college[
    ["draft_row_id", "DraftYear", "Pk", "Player_clean"]
].merge(
    college,
    on="Player_clean",
    how="left"
)

# Keep only seasons before or during the draft year
college_candidates = college_candidates[
    college_candidates["year"].notna()
    & (college_candidates["year"] <= college_candidates["DraftYear"])
].copy()

college_candidates["year_gap"] = (
    college_candidates["DraftYear"] - college_candidates["year"]
)

# Helpful tie-breaker when names repeat
college_candidates["college_pick_num"] = pd.to_numeric(
    college_candidates["pick"], errors="coerce"
)

college_candidates["draft_pick_num"] = pd.to_numeric(
    college_candidates["Pk"], errors="coerce"
)

college_candidates["pick_match"] = (
    college_candidates["college_pick_num"].round().astype("Int64")
    .eq(college_candidates["draft_pick_num"].astype("Int64"))
    .fillna(False)
    .astype(bool)
)

# Pick ONE college row per drafted player:
# 1. prefer matching draft pick
# 2. prefer closest season to draft year
# 3. prefer bigger-minute season if still tied
college_candidates = college_candidates.sort_values(
    ["draft_row_id", "pick_match", "year_gap", "min", "g"],
    ascending=[True, False, True, False, False]
)

college_best = college_candidates.drop_duplicates("draft_row_id").copy()

Candidate seasons were prioritized using the following criteria:

1. Whether the college record matched the player's draft pick number.
2. How close the college season was to the player's draft year.
3. Total minutes played during the season.
4. Total games played during the season.

After ranking all possible matches, only the highest-priority record for each drafted player was retained. This approach links every prospect to the college season most representative of their pre-draft performance while preventing duplicate player records in the final dataset.

In [59]:
college_candidates = college_candidates.sort_values(
    ["draft_row_id", "pick_match", "year_gap", "min", "g"],
    ascending=[True, False, True, False, False]
)

college_best = college_candidates.drop_duplicates("draft_row_id").copy()

After selecting one college season per drafted player, I kept the most relevant college performance columns, renamed them with a `college_` prefix, and left-merged them into the master dataset using `draft_row_id`. This preserves every drafted player in the dataset while adding college statistics when available. Players without matched college data, such as international players or players from non-college pathways, remain in the dataset with missing college values.

In [63]:
college_cols = [
    "draft_row_id",
    "year",
    "year_gap",
    "player",
    "team",
    "conf",
    "pos",
    "exp",
    "g",
    "min",
    "porpag",
    "dporpag",
    "ortg",
    "drtg",
    "obpm",
    "dbpm",
    "bpm",
    "oreb",
    "dreb",
    "ast",
    "to",
    "blk",
    "stl",
    "ftr",
    "pfr",
    "mpg",
    "ppg",
    "fg_pct",
    "rpg",
    "apg",
    "tov",
    "ast_to",
    "spg",
    "bpg",
    "usg",
    "efg",
    "ts",
    "ft_pct",
    "two_pct",
    "three_pct",
    "rim_pct",
    "mid_pct",
]

college_features = college_best[college_cols].copy()

college_features = college_features.rename(
    columns={
        col: f"college_{col}"
        for col in college_features.columns
        if col != "draft_row_id"
    }
)

# Delete any old college stats that might already be there
old_college_cols = [
    col for col in master.columns
    if col.startswith("college_") or col == "has_college_stats"
]

master = master.drop(columns=old_college_cols, errors="ignore")

master = master.merge(
    college_features,
    on="draft_row_id",
    how="left"
)

master["has_college_stats"] = master["college_player"].notna()

After merging draft, combine, and college statistics into the master dataset, I created a final Tableau-ready CSV. Blank draft rows from Basketball Reference were removed so each row represents one real drafted player. I also added indicator columns for whether each player has combine data or college statistics, which will make it easier to filter and interpret missing data in Tableau.

Several columns were renamed to make the dataset easier to use in visualizations. For example, NBA per-game statistics were renamed as `nba_ppg`, `nba_rpg`, `nba_apg`, and `nba_mpg`, while combine fields were renamed with clearer labels. The final dataset is exported as `tableau_nba_draft_master.csv`, which will be used for Tableau dashboards and player comparisons.

In [64]:
tableau_data = master.copy()

# Remove blank Basketball Reference draft rows
tableau_data = tableau_data[tableau_data["Player_draft"].notna()].copy()

# Helpful flags for Tableau filters
tableau_data["has_combine_data"] = tableau_data["uniqueID"].notna()
tableau_data["has_college_stats"] = tableau_data["college_player"].notna()

# Optional cleaner player column
tableau_data = tableau_data.rename(columns={
    "Player_draft": "player",
    "Player_combine": "combine_player",
    "Position": "combine_position",
    "Height in Ft & In": "combine_height",
    "Weight in lbs": "combine_weight_lbs",
    "PTS.1": "nba_ppg",
    "TRB.1": "nba_rpg",
    "AST.1": "nba_apg",
    "MP.1": "nba_mpg",
    "FG%": "nba_fg_pct",
    "3P%": "nba_3p_pct",
    "FT%": "nba_ft_pct",
})

print(tableau_data.shape)
print(tableau_data["has_combine_data"].mean())
print(tableau_data["has_college_stats"].mean())

tableau_data.to_csv("tableau_nba_draft_master.csv", index=False)

(953, 80)
0.6935991605456453
0.48478488982161594
